In [10]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

In [11]:
print(df_train.columns)

Index(['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber',
       'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [14]:
df_train_merged["Race"].unique()

array(['Mexico City Grand Prix', 'Italian Grand Prix',
       'Monaco Grand Prix', 'Azerbaijan Grand Prix',
       'São Paulo Grand Prix', 'Emilia Romagna Grand Prix',
       'Canadian Grand Prix', 'Chinese Grand Prix',
       'Singapore Grand Prix', 'Hungarian Grand Prix',
       'French Grand Prix', 'Abu Dhabi Grand Prix', 'Austrian Grand Prix',
       'Japanese Grand Prix', 'Belgian Grand Prix', 'British Grand Prix',
       'Dutch Grand Prix', 'United States Grand Prix', 'Miami Grand Prix',
       'Spanish Grand Prix', 'Saudi Arabian Grand Prix',
       'Las Vegas Grand Prix', 'Qatar Grand Prix',
       'Australian Grand Prix', 'Bahrain Grand Prix',
       'Pre-Season Testing', 'Pre-Season Track Session',
       'Pre-Season Test'], dtype=object)

In [15]:
df_train_merged.columns

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap'],
      dtype='object')

In [16]:
df_train_merged[["Compound"]].value_counts()

Compound    
MEDIUM          248780
HARD            215485
SOFT             51488
INTERMEDIATE     22938
WET               1754
Name: count, dtype: int64

In [17]:
#Tire dataset
def add_tire_life_columns(df):
    max_tire_life = {
        "MEDIUM": 40,
        "HARD": 55,
        "SOFT": 30,
        "INTERMEDIATE": 25,
        "WET": 20
    }

    df["MaxTireLife"] = df["Compound"].map(max_tire_life)

    df["TireRemainingLife"] = (
        df["MaxTireLife"] - df["TyreLife"]
    ).clip(lower=0)
    df["TireAgePct"] = (
    df["TyreLife"] / df["MaxTireLife"]).clip(0, 1)
    df["TireRemainingPct"] = (df["TireRemainingLife"] / df["MaxTireLife"]).clip(0, 1)
    

    return df



In [18]:
df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [19]:
def add_laps_for_each_race(df):
    df["RaceLaps"] = (
    df["LapNumber"] / (df["RaceProgress"])
).round()
    df["LapsRemaining"] = df["RaceLaps"] - df["LapNumber"]
    return df


In [20]:
def prev_position_calcute(df):
    df["Prev_Position"] = df["Position"] + df["Position_Change"]
    return df


In [21]:
def get_trend_data(df):
    df["LapTimeAvg_3"] = (
        df.groupby(["Year", "Race", "Driver"])["LapTime (s)"]
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )
    return df


In [22]:
def set_fe(df):
    df = add_tire_life_columns(df)
    df = add_laps_for_each_race(df)
    df= prev_position_calcute(df)
    df = get_trend_data(df)
    return df

In [23]:
df_train_merged = set_fe(df_train_merged)
df_train=set_fe(df_train)
df_test=set_fe(df_test)

In [24]:
df_train =df_train.drop("id",axis=1)
df_test =df_test.drop("id",axis=1)

In [152]:
df_train_merged.isnull().sum()

Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
MaxTireLife               0
TireRemainingLife         0
TireAgePct                0
TireRemainingPct          0
RaceLaps                  0
LapsRemaining             0
Prev_Position             0
LapTimeAvg_3              0
dtype: int64

In [153]:
df_train.isnull().sum()

Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
MaxTireLife               0
TireRemainingLife         0
TireAgePct                0
TireRemainingPct          0
RaceLaps                  0
LapsRemaining             0
Prev_Position             0
LapTimeAvg_3              0
dtype: int64

In [154]:
df_test.isnull().sum()

Driver                    0
Compound                  0
Race                      0
Year                      0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
MaxTireLife               0
TireRemainingLife         0
TireAgePct                0
TireRemainingPct          0
RaceLaps                  0
LapsRemaining             0
Prev_Position             0
LapTimeAvg_3              0
dtype: int64

In [7]:
from autogluon.tabular import TabularDataset, TabularPredictor
TARGET = 'PitNextLap'

print(df_train[TARGET].value_counts())
print(df_train_merged[TARGET].value_counts())

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    427273
1.0    113172
Name: count, dtype: int64


In [156]:
print(df_train.columns)
print(df_train.columns)

Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap', 'MaxTireLife', 'TireRemainingLife', 'TireAgePct',
       'TireRemainingPct', 'RaceLaps', 'LapsRemaining', 'Prev_Position',
       'LapTimeAvg_3'],
      dtype='object')
Index(['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint',
       'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta',
       'Cumulative_Degradation', 'RaceProgress', 'Position_Change',
       'PitNextLap', 'MaxTireLife', 'TireRemainingLife', 'TireAgePct',
       'TireRemainingPct', 'RaceLaps', 'LapsRemaining', 'Prev_Position',
       'LapTimeAvg_3'],
      dtype='object')


In [157]:
import warnings
warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*downcast.*"
)
_orig_fillna = pd.DataFrame.fillna
def _patched_fillna(self, *args, **kwargs):
    kwargs.pop("downcast", None)
    return _orig_fillna(self, *args, **kwargs)

In [158]:
models = {
    "GBM": [
        {},  # Generates LightGBM_BAG_L1
        {"extra_trees": True, "ag_args": {"name_suffix": "Large"}},  # Generates LightGBMLarge_BAG_L1
    ],
    "XGB": {},  # Generates XGBoost_BAG_L1
    "CAT": {},  # Generates CatBoost_BAG_L1
}


In [160]:
predictor = TabularPredictor(label=TARGET,eval_metric='roc_auc', path='ag_models4').fit(
    train_data=df_train_merged,
    ag_args_fit={"num_gpus": 1},
    time_limit=3600*9,
    presets='best_quality',
    verbosity=3,
    num_stack_levels=0,
    hyperparameters=models,
)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.8.0+cu129
CUDA Version:       12.9
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       4.55 GB / 15.06 GB (30.2%)
Disk Space Avail:   666.86 GB / 930.47 GB (71.7%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 0,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 '

[50]	valid_set's binary_logloss: 0.283323
[100]	valid_set's binary_logloss: 0.260562
[150]	valid_set's binary_logloss: 0.251288
[200]	valid_set's binary_logloss: 0.246148
[250]	valid_set's binary_logloss: 0.242052
[300]	valid_set's binary_logloss: 0.239459
[350]	valid_set's binary_logloss: 0.237322
[400]	valid_set's binary_logloss: 0.235122
[450]	valid_set's binary_logloss: 0.233585
[500]	valid_set's binary_logloss: 0.23242
[550]	valid_set's binary_logloss: 0.231263
[600]	valid_set's binary_logloss: 0.230295
[650]	valid_set's binary_logloss: 0.229496
[700]	valid_set's binary_logloss: 0.228779
[750]	valid_set's binary_logloss: 0.228188
[800]	valid_set's binary_logloss: 0.227512
[850]	valid_set's binary_logloss: 0.226951
[900]	valid_set's binary_logloss: 0.226497
[950]	valid_set's binary_logloss: 0.226007
[1000]	valid_set's binary_logloss: 0.225526
[1050]	valid_set's binary_logloss: 0.225237
[1100]	valid_set's binary_logloss: 0.224949
[1150]	valid_set's binary_logloss: 0.224708
[1200]	va

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


[50]	valid_set's binary_logloss: 0.284803
[100]	valid_set's binary_logloss: 0.262136
[150]	valid_set's binary_logloss: 0.253508
[200]	valid_set's binary_logloss: 0.248086
[250]	valid_set's binary_logloss: 0.244479
[300]	valid_set's binary_logloss: 0.242022
[350]	valid_set's binary_logloss: 0.239586
[400]	valid_set's binary_logloss: 0.237857
[450]	valid_set's binary_logloss: 0.236703
[500]	valid_set's binary_logloss: 0.235129
[550]	valid_set's binary_logloss: 0.234002
[600]	valid_set's binary_logloss: 0.233101
[650]	valid_set's binary_logloss: 0.232317
[700]	valid_set's binary_logloss: 0.231694
[750]	valid_set's binary_logloss: 0.230995
[800]	valid_set's binary_logloss: 0.230439
[850]	valid_set's binary_logloss: 0.230029
[900]	valid_set's binary_logloss: 0.229363
[950]	valid_set's binary_logloss: 0.228919
[1000]	valid_set's binary_logloss: 0.228542
[1050]	valid_set's binary_logloss: 0.228239
[1100]	valid_set's binary_logloss: 0.227949
[1150]	valid_set's binary_logloss: 0.227751
[1200]	v

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


...

	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.47466
[50]	validation_0-logloss:0.26443
[100]	validation_0-logloss:0.24781
[150]	validation_0-logloss:0.23996
[200]	validation_0-logloss:0.23398
[250]	validation_0-logloss:0.23036
[300]	validation_0-logloss:0.22776
[350]	validation_0-logloss:0.22543
[400]	validation_0-logloss:0.22350
[450]	validation_0-logloss:0.22147
[500]	validation_0-logloss:0.22022
[550]	validation_0-logloss:0.21904
[600]	validation_0-logloss:0.21782
[650]	validation_0-logloss:0.21686
[700]	validation_0-logloss:0.21616
[750]	validation_0-logloss:0.21532
[800]	validation_0-logloss:0.21416
[850]	validation_0-logloss:0.21368
[900]	validation_0-logloss:0.21302
[950]	validation_0-logloss:0.21244
[1000]	validation_0-logloss:0.21194
[1050]	validation_0-logloss:0.21163
[1100]	validation_0-logloss:0.21139
[1150]	validation_0-logloss:0.21109
[1200]	validation_0-logloss:0.21088
[1250]	validation_0-logloss:0.21051
[1300]	validation_0-logloss:0.21032
[1350]	validation_0-logloss:0.21018
[1400]	validati

	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.47463
[50]	validation_0-logloss:0.26261
[100]	validation_0-logloss:0.24656
[150]	validation_0-logloss:0.23869
[200]	validation_0-logloss:0.23268
[250]	validation_0-logloss:0.22892
[300]	validation_0-logloss:0.22598
[350]	validation_0-logloss:0.22332
[400]	validation_0-logloss:0.22158
[450]	validation_0-logloss:0.22002
[500]	validation_0-logloss:0.21854
[550]	validation_0-logloss:0.21741
[600]	validation_0-logloss:0.21596
[650]	validation_0-logloss:0.21506
[700]	validation_0-logloss:0.21430
[750]	validation_0-logloss:0.21365
[800]	validation_0-logloss:0.21292
[850]	validation_0-logloss:0.21226
[900]	validation_0-logloss:0.21174
[950]	validation_0-logloss:0.21121
[1000]	validation_0-logloss:0.21073
[1050]	validation_0-logloss:0.21027
[1100]	validation_0-logloss:0.21002
[1150]	validation_0-logloss:0.20972
[1200]	validation_0-logloss:0.20935
[1250]	validation_0-logloss:0.20916
[1300]	validation_0-logloss:0.20897
[1350]	validation_0-logloss:0.20880
[1400]	validati

Saving c:\Darshak\Projects\Hackathon\ag_models4\models\XGBoost_BAG_L1\utils\oof.pkl
Saving c:\Darshak\Projects\Hackathon\ag_models4\models\XGBoost_BAG_L1\model.pkl
	0.9587	 = Validation score   (roc_auc)
	613.84s	 = Training   runtime
	4.05s	 = Validation runtime
	16664.4	 = Inference  throughput (rows/s | 67556 batch size)
Saving c:\Darshak\Projects\Hackathon\ag_models4\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models4\models\LightGBM_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models4\models\LightGBMLarge_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models4\models\CatBoost_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models4\models\XGBoost_BAG_L1\utils\oof.pkl
Model configs that will be trained (in order):
	WeightedEnsemble_L2: 	{'ag_args': {'problem_types': ['binary', 'multiclass', 'regression', 'quantile', 'softclass'], 'valid_base': False, 'name_bag_suffix': '', 'model_type': <class 'autogluon.core.models.gr

In [161]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.960512,roc_auc,25.226204,1758.618675,0.050089,5.023120,2,True,5
1,CatBoost_BAG_L1,0.959010,roc_auc,3.165922,829.781863,3.165922,829.781863,1,True,3
2,XGBoost_BAG_L1,0.958686,roc_auc,4.053901,613.844760,4.053901,613.844760,1,True,4
3,LightGBMLarge_BAG_L1,0.953884,roc_auc,12.682142,203.142475,12.682142,203.142475,1,True,2
4,LightGBM_BAG_L1,0.951644,roc_auc,5.274150,106.826458,5.274150,106.826458,1,True,1


In [8]:
predictor = TabularPredictor.load(f"ag_models4")

In [25]:
df=predictor.predict_proba(df_test)
df.head()

,0,1
0,0.994774,0.005226
1,0.992815,0.007185
2,0.994747,0.005253
3,0.881715,0.118285
4,0.090523,0.909477


In [163]:
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [164]:
df_sample_out['PitNextLap']=df[1]

In [165]:
df_sample_out.to_csv("My_output/all_model_together_best_fe_1.csv", index=False)

In [ ]:
#Try without driver

In [166]:
df_train_merged_wo_driver = df_train_merged.drop("Driver", axis=1)
df_train_wo_driver = df_train.drop("Driver", axis=1)
df_test_wo_driver = df_test.drop("Driver", axis=1)

In [ ]:
predictor_wo_driver = TabularPredictor(label=TARGET,eval_metric='roc_auc', path='ag_models8').fit(
    train_data=df_train_merged_wo_driver,
    ag_args_fit={"num_gpus": 1},
    time_limit=3600*9,
    presets='best_quality',
    verbosity=3,
    num_stack_levels=0,
    hyperparameters=models,
)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.8.0+cu129
CUDA Version:       12.9
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       4.30 GB / 15.06 GB (28.6%)
Disk Space Avail:   663.17 GB / 930.47 GB (71.3%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 0,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 '

[50]	valid_set's binary_logloss: 0.28329
[100]	valid_set's binary_logloss: 0.258729
[150]	valid_set's binary_logloss: 0.248807
[200]	valid_set's binary_logloss: 0.243726
[250]	valid_set's binary_logloss: 0.240096
[300]	valid_set's binary_logloss: 0.237042
[350]	valid_set's binary_logloss: 0.234653
[400]	valid_set's binary_logloss: 0.232601
[450]	valid_set's binary_logloss: 0.230833
[500]	valid_set's binary_logloss: 0.22939
[550]	valid_set's binary_logloss: 0.228018
[600]	valid_set's binary_logloss: 0.226967
[650]	valid_set's binary_logloss: 0.225923
[700]	valid_set's binary_logloss: 0.225019
[750]	valid_set's binary_logloss: 0.22409
[800]	valid_set's binary_logloss: 0.223413
[850]	valid_set's binary_logloss: 0.222513
[900]	valid_set's binary_logloss: 0.221685
[950]	valid_set's binary_logloss: 0.220989
[1000]	valid_set's binary_logloss: 0.22048
[1050]	valid_set's binary_logloss: 0.220011
[1100]	valid_set's binary_logloss: 0.219554
[1150]	valid_set's binary_logloss: 0.219099
[1200]	valid

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


[50]	valid_set's binary_logloss: 0.285085
[100]	valid_set's binary_logloss: 0.260159
[150]	valid_set's binary_logloss: 0.249919
[200]	valid_set's binary_logloss: 0.24446
[250]	valid_set's binary_logloss: 0.240438
[300]	valid_set's binary_logloss: 0.237546
[350]	valid_set's binary_logloss: 0.235152
[400]	valid_set's binary_logloss: 0.233362
[450]	valid_set's binary_logloss: 0.231823
[500]	valid_set's binary_logloss: 0.230332
[550]	valid_set's binary_logloss: 0.228916
[600]	valid_set's binary_logloss: 0.227803
[650]	valid_set's binary_logloss: 0.226619
[700]	valid_set's binary_logloss: 0.225739
[750]	valid_set's binary_logloss: 0.224895
[800]	valid_set's binary_logloss: 0.223947
[850]	valid_set's binary_logloss: 0.223155
[900]	valid_set's binary_logloss: 0.222446
[950]	valid_set's binary_logloss: 0.221821
[1000]	valid_set's binary_logloss: 0.22103
[1050]	valid_set's binary_logloss: 0.220512
[1100]	valid_set's binary_logloss: 0.21986
[1150]	valid_set's binary_logloss: 0.219265
[1200]	vali

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


...

	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.47462
[50]	validation_0-logloss:0.26407
[100]	validation_0-logloss:0.24778
[150]	validation_0-logloss:0.23980
[200]	validation_0-logloss:0.23436
[250]	validation_0-logloss:0.23077
[300]	validation_0-logloss:0.22777
[350]	validation_0-logloss:0.22587
[400]	validation_0-logloss:0.22379
[450]	validation_0-logloss:0.22224
[500]	validation_0-logloss:0.22126
[550]	validation_0-logloss:0.22014
[600]	validation_0-logloss:0.21905
[650]	validation_0-logloss:0.21815
[700]	validation_0-logloss:0.21740
[750]	validation_0-logloss:0.21677
[800]	validation_0-logloss:0.21632
[850]	validation_0-logloss:0.21581
[900]	validation_0-logloss:0.21552
[950]	validation_0-logloss:0.21506
[1000]	validation_0-logloss:0.21471
[1050]	validation_0-logloss:0.21445
[1100]	validation_0-logloss:0.21425
[1150]	validation_0-logloss:0.21411
[1200]	validation_0-logloss:0.21383
[1250]	validation_0-logloss:0.21369
[1300]	validation_0-logloss:0.21353
[1350]	validation_0-logloss:0.21332
[1400]	validati

	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.47463
[50]	validation_0-logloss:0.26295
[100]	validation_0-logloss:0.24648
[150]	validation_0-logloss:0.23843
[200]	validation_0-logloss:0.23284
[250]	validation_0-logloss:0.22911
[300]	validation_0-logloss:0.22644
[350]	validation_0-logloss:0.22413
[400]	validation_0-logloss:0.22246
[450]	validation_0-logloss:0.22087
[500]	validation_0-logloss:0.21990
[550]	validation_0-logloss:0.21874
[600]	validation_0-logloss:0.21792
[650]	validation_0-logloss:0.21725
[700]	validation_0-logloss:0.21651
[750]	validation_0-logloss:0.21597
[800]	validation_0-logloss:0.21558
[850]	validation_0-logloss:0.21527
[900]	validation_0-logloss:0.21477
[950]	validation_0-logloss:0.21449
[1000]	validation_0-logloss:0.21397
[1050]	validation_0-logloss:0.21382
[1100]	validation_0-logloss:0.21337
[1150]	validation_0-logloss:0.21324
[1200]	validation_0-logloss:0.21307
[1250]	validation_0-logloss:0.21285
[1300]	validation_0-logloss:0.21263
[1350]	validation_0-logloss:0.21252
[1400]	validati

Saving c:\Darshak\Projects\Hackathon\ag_models8\models\XGBoost_BAG_L1\utils\oof.pkl
Saving c:\Darshak\Projects\Hackathon\ag_models8\models\XGBoost_BAG_L1\model.pkl
	0.9571	 = Validation score   (roc_auc)
	273.06s	 = Training   runtime
	2.97s	 = Validation runtime
	22732.5	 = Inference  throughput (rows/s | 67556 batch size)
Saving c:\Darshak\Projects\Hackathon\ag_models8\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\LightGBM_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\LightGBMLarge_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\CatBoost_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\XGBoost_BAG_L1\utils\oof.pkl
Model configs that will be trained (in order):
	WeightedEnsemble_L2: 	{'ag_args': {'problem_types': ['binary', 'multiclass', 'regression', 'quantile', 'softclass'], 'valid_base': False, 'name_bag_suffix': '', 'model_type': <class 'autogluon.core.models.gr

In [169]:
leaderboard = predictor_wo_driver.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.959125,roc_auc,37.932610,1384.518738,0.050312,5.060219,2,True,5
1,LightGBM_BAG_L1,0.958277,roc_auc,11.543802,210.660673,11.543802,210.660673,1,True,1
2,XGBoost_BAG_L1,0.957086,roc_auc,2.971786,273.060256,2.971786,273.060256,1,True,4
3,LightGBMLarge_BAG_L1,0.956886,roc_auc,22.951177,296.346481,22.951177,296.346481,1,True,2
4,CatBoost_BAG_L1,0.954769,roc_auc,0.415533,599.391110,0.415533,599.391110,1,True,3


In [170]:
df=predictor_wo_driver.predict_proba(df_test)
df.head()
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()
df_sample_out['PitNextLap']=df[1]
df_sample_out.to_csv("My_output/all_model_together_best_fe_without_driver_1.csv", index=False)

Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\CatBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\LightGBM_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models8\models\WeightedEnsemble_L2\model.pkl
